# W8A8 FP8 linears on compile + Sage fp16-PV: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_fp8_2026-11-18.md` (F1–F9).

Time: about 10 min to build SageAttention, then about 1 h 30 min of runs (~3.3 compute units). No cell waits for input. Runs are ordered by importance.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit,driver_version --format=csv

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.2 accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)
!nvcc --version | tail -1

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## Build SageAttention for sm_89 (~10 min)

Same build and commit as before. Stops the notebook if it fails.

In [ ]:
import os, re, subprocess, sys, torch
maj, mi = torch.version.cuda.split('.')[:2]
m = re.search(r'release (\d+)\.(\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
print('torch CUDA', torch.version.cuda, '| nvcc', m.group(0) if m else 'not found')
if not m or m.group(1) != maj:
    ubuntu = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True, capture_output=True,
                            text=True, executable='/bin/bash').stdout.strip()
    v = f'{maj}-{mi}'
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/{ubuntu}/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get -qq update > /dev/null && apt-get -qq install -y cuda-minimal-build-{v} cuda-cudart-dev-{v} cuda-cccl-{v} cuda-nvrtc-dev-{v} libcublas-dev-{v} libcusparse-dev-{v} libcusolver-dev-{v} libcurand-dev-{v} > /dev/null
    os.environ['CUDA_HOME'] = f'/usr/local/cuda-{maj}.{mi}'
    os.environ['PATH'] = os.environ['CUDA_HOME'] + '/bin:' + os.environ['PATH']
    !nvcc --version | tail -2

%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention checkout -q d1a57a5  # the commit measured in results/L4_sage.md
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
# torch >= 2.10 headers use C++20 `requires` clauses; SageAttention's setup.py hardcodes -std=c++17
!sed -i 's/-std=c++17/-std=c++20/g' setup.py && grep -n 'std=c++' setup.py && g++ --version | head -1
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation -v . > /content/sage_build.log 2>&1; tail -40 /content/sage_build.log
%cd /content/video-dit-l4
r = subprocess.run([sys.executable, '-c', 'import importlib.metadata as m, sageattention; print("sageattention", m.version("sageattention"))'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('SageAttention did not build or import; stopping here. Send the output above.')

## FP8 GEMM check and microbenchmark (F1–F3, ~3 min)

In [ ]:
!python bench/fp8_linear.py
!python bench/fp8_micro.py --out results/fp8_micro.json

## 33 frames, 4 prompts

Base without FP8 (~12 min), with the kernel split. F9 checks it against two earlier sessions.

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --kernels --out results/f_base_33f
!python bench/hash_frames.py results/f_base_33f > /dev/null
import json
a = json.load(open('results/f_base_33f/frames_sha256.json'))
for ref, name in [('pv16_33f', 'frames_{}.npy'), ('q_steps50', 'frames_{}_s0.npy')]:
    b = json.load(open(f'results/{ref}/frames_sha256.json'))
    same = sum(a[f'frames_{p}.npy']['sha256'] == b[name.format(p)]['sha256'] for p in range(4))
    print(f'base vs {ref}: {same} of 4 identical')

FP8 rowwise (~11 min).

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --fp8-linears rowwise --kernels --out results/f_fp8row_33f
!python bench/compare_frames.py --ref results/f_base_33f --test results/f_fp8row_33f

fp32 reference, fox and market (~30 min), then every 33 f run against it (F6, F7).

In [ ]:
!python bench/profile_baseline.py --frames 33 --dtype fp32 --attn efficient --prompt-ids 0 3 --out results/f_fp32_33f
!python bench/hash_frames.py results/f_fp32_33f --check results/fp32_33f/frames_sha256.json
for t in ['f_base_33f', 'f_fp8row_33f']:
    !python bench/compare_frames.py --ref results/f_fp32_33f --test results/{t} --out-name compare_vs_fp32.json

FP8 tensorwise (~11 min).

In [ ]:
!python bench/profile_baseline.py --frames 33 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --fp8-linears tensorwise --out results/f_fp8tensor_33f
!python bench/compare_frames.py --ref results/f_base_33f --test results/f_fp8tensor_33f
!python bench/compare_frames.py --ref results/f_fp32_33f --test results/f_fp8tensor_33f --out-name compare_vs_fp32.json

## 81 frames, fox (~21 min), F4

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --kernels --out results/f_base_81f
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --compile blocks --attn-backend _sage_qk_int8_pv_fp16_cuda --fp8-linears rowwise --kernels --out results/f_fp8row_81f
!python bench/compare_frames.py --ref results/f_base_81f --test results/f_fp8row_81f

In [ ]:
# Hashes of every run; zip without .npy
import glob, os
runs = sorted(d for d in glob.glob('results/f_*') if os.path.isdir(d))
for d in runs:
    !python bench/hash_frames.py {d} > /dev/null
runs_str = ' '.join(runs)
!zip -qr fp8_results.zip {runs_str} results/fp8_micro.json -x '*.npy' && ls -lh fp8_results.zip
from google.colab import files; files.download('fp8_results.zip')